<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/10-ml-system-design/01-designing-an-inference-service.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 设计一个推理服务（概念篇）

**目标：** 完整走一遍 ML 系统设计面试。把「设计一个 LLM 功能」这种模糊的题目，变成一个定好容量、经得起追问的服务化系统：需求、信封背面的粗算数字（QPS / VRAM / 延迟 / 成本）、架构，以及你在被追问时要守住的那些取舍。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

> **这是一个概念 notebook，没有代码可跑。** 系统设计是一场*对话*，不是一个程序：交付物是一段结构化的口头回答和一块白板，所以本篇讲的是方法，外加一个完整推演的示例。它依赖的数学就是你在 [09/02](../09-serving-inference/02-inference-performance.zh.ipynb) 已经跑过的定容量算术；这里你是在面试压力下*运用*它。

## 为什么 AI 工程师会有系统设计轮

前面每一节都在搭一个*组件*：一条 RAG 管线、一个 agent 循环、一个评测骨架。这一轮问的是下一个问题：**你能把它部署成一个扛得住真实流量、预算和 SLA 的服务吗？** 就是这一轮把「我在 notebook 里调用过一个 API」和「我能负责一个 AI 产品的服务层」区分开来——这也是本路线瞄准的 AI 分布式系统岗位。

好消息是：每一块料你都已经有了。服务化框架（09/01）、吞吐/延迟杠杆与定容量数学（09/02）、可靠性模式（08/02）、评测（02/04）、成本管控（00/01）。本篇是**连接组织**：把它们按什么顺序摆出来，以及它们如何拼成一个站得住的设计。

> **⭐ 关键结论 ——** 面试官打分的不是*答案*，而是*方法*。一个说「先让我把需求问清楚、把数算出来」并给出一个*粗略*数字的候选人，胜过那个把 Kubernetes 挂在嘴边却从不做任何定容量的。结构才是本事。

## 方法 —— 一个五步循环

任何「设计一个推理服务」的题目都用这个骨架。它刻意与 09/02 的定容量动作同形，只是展开成一份完整设计。

1. **界定范围与需求：** 设计之前先把问题钉死。功能性需求（它做什么？）和非功能性需求（**延迟目标、QPS、成本上限、质量门槛**）。*像做一次 discovery 通话那样推进（第 11 节）：模糊的题干是故意的。*
2. **信封背面的粗算：** QPS → tokens/sec → GPU 数；每 GPU 的 VRAM；粗略的每月成本。先有数字，再有框图。
3. **架构：** 框和箭头：client → gateway → queue → model servers → cache，外加评测和可观测性在哪里接入。
4. **取舍与瓶颈：** 点出最先崩的是什么，以及你会拉哪个杠杆（这就是 09/02 的回报）。
5. **应对追问、持续迭代。** 面试官会加压（「流量 ×10」、「成本砍一半」、「p99 飙了」）；每一个都是一个杠杆，不是一次重新设计。

后面的内容会用一道题走完这五步。

## 题目

> *「为一个 AI 编程助手设计服务后端。开发者在 IDE 里敲代码，得到行内代码补全。我们大约有 10,000 名日活用户。开始吧。」*

刻意写得不够具体：这就是考点。**第 1 步是盘问它**，而不是开始画图。

## 第 1 步 —— 界定范围与需求（去问，别假设）

你会问的问题，以及如果面试官摆手让你跳过时，你会说明并继续往下走的假设：

- **交互式还是批处理？** 行内补全 → *交互式*。**TTFT 才是关键指标**（开发者正盯着光标）。假设一个目标：**p95 TTFT < 500 ms**。
- **流量到底有多大？** 1 万 DAU 不等于 1 万 QPS。假设每个活跃开发者在约 3 小时的峰值窗口里触发约 20 次补全/小时 → 在第 2 步估算峰值 **QPS**。
- **输出有多长？** 行内补全很短：假设**约 50 个输出 token**，prompt（周围代码）**约 1,000 token**。这一项主导 KV cache 的算术。
- **质量门槛和模型？** 假设一个代码微调过的模型，能放进一张现代 GPU（例如 fp16 的 7B，或更大的量化版）。*模型选择是质量与成本的杠杆，不是第一个决策。*
- **成本上限？** 永远要问。假设「GPU 开销保持合理」→ 我们会报一个数字，并给出一个能砍掉它的旋钮。

> **🔵 面试信号 ——** 一上来就问*「交互式还是批处理，延迟目标是多少？」*，正是 09/02 训练出来的本能。它告诉面试官：你知道吞吐/延迟取舍支配着每一个下游选择。

## 第 2 步 —— 信封背面的粗算（那些数字）

这就是 09/02 的算术，派上用场。大声算出来，四舍五入随意。

**峰值 QPS。** 1 万 DAU，峰值那一小时约 ⅓ 活跃（约 3.3k），每人约 20 次补全/小时 ≈ 66k/小时 ≈ **平均约 18 QPS**，所以假设**峰值约 40 QPS**（突发性强，开发者扎堆）。取整为 **40 QPS**。

**吞吐需求。** 40 QPS × 50 个输出 token = **2,000 输出 tok/s**。（1k token 的 prompt 的 prefill 会影响 TTFT，但决定稳态吞吐的是 decode 的 token。）

**GPU 数。** 由 09/02：`GPU 数 = 需求 ÷（每 GPU 吞吐 × 利用率）`。假设跑代码模型的 GPU 能到**约 2,500 批处理 tok/s**，我们跑在 **70% 利用率**以留余量 → 2,000 ÷ (2,500 × 0.7) ≈ **约 1.15 → 2 张 GPU** 用于算力。取整**向上**，而且为了冗余你本来也至少会跑 2 张。

**VRAM 合理性检查。** 权重（7B fp16 ≈ 14 GB）+ KV cache。由 09/02，1k 上下文的请求很轻（约 0.5 GB），所以一张 24–40 GB 的 GPU 能批很多。**这里不受显存限制**，很好：算力/QPS 才是真正的约束。

**成本。** 约 2 张 GPU × 约 $2/GPU-小时 × 730 小时 ≈ 这个规模下**约 $3k/月**，还没算非峰值时段自动缩容，缩容后大致能砍一半。*把它说出来；这才是业务关心的数字。*

> **⚠️ 生产现实 ——** 这些数字是*故意粗略的*，而且你要明说。要点在于量级正确，以及知道**每个输出对哪个输入敏感**（输出长度减半 → GPU 数减半；用户 ×10 → GPU ×10）。精度来自压测，而不是白板。

## 第 3 步 —— 架构（框和箭头）

现在画图，每个框都能追溯到本仓库的某一节：

```
 IDE 客户端 ─▶ API 网关 ─▶ 请求队列 ─▶ ┌─ model server (vLLM) ─┐
 （去重、         （鉴权、     （削平         │  ...replica 2         │ ─▶ 补全
  防抖）           限流）       突发）        └─ ...replica N (HPA) ──┘
                       │                              ▲        │
                   prompt 缓存 ──────────────────────┘        ▼
                   （重复前缀）                     可观测性 + 评测
                                                    （08/01, 02/04）
```

- **IDE 客户端：** 对按键做防抖，并取消已被取代的请求。*最便宜的容量，是你根本没发出去的那个请求。*
- **API 网关：** 鉴权、限流、路由。这是稳定的缝隙；在它后面，你可以不动调用方就换掉模型层（09/01）。
- **请求队列：** 吸收第 2 步标出的突发。交互式流量是尖刺状的；正是队列让 p95 TTFT 在尖峰下仍然诚实，而不是直接丢请求。
- **模型服务器（vLLM），N 个副本：** 第 2 步算出的那约 2 张 GPU，放在自动扩缩器后面（例如按队列深度 / GPU 利用率触发的 HPA）。连续批处理就发生在这里（09/02）。
- **prompt 缓存：** 代码补全的 prompt 共享巨大的前缀（同一个文件、imports）。缓存重复前缀能减少 prefill 工作量 → 更好的 TTFT 和吞吐。
- **可观测性 + 评测：** 追踪每次调用（08/01）；在模型/量化变更*上线之前*，跑离线评测集（02/04）。

## 第 4 步 —— 取舍与瓶颈（最先崩的是什么）

在面试官之前点出这些失效模式。这就是资深信号。

- **第一个瓶颈：突发下的 TTFT。** 开发者扎堆时，批处理会抬高单请求延迟（09/02 的取舍）。缓解：为这个交互式服务限制 batch size、保留利用率余量、按队列深度扩容，并接受略高一点的 GPU 账单来保住 TTFT。
- **成本与质量的旋钮：模型。** 更大/更高质量的模型会抬高 TTFT 和 GPU 数。量化模型（09/02）能同时削减两者，代价是质量，而你会用 02/04 的评测去*度量*它。这是主旋钮，你要把它作为一个点明确说出来。
- **成本与延迟：自动扩缩容。** 非峰值时段缩到零能省钱，但会增加冷启动延迟。对一个有明确工作时间的开发工具，夜里*降*容（但不到零）：这是站得住脚的折中。
- **可靠性（08/02）：** 一个副本挂了、供应商抽了一下。健康检查 + 队列 + N≥2 副本，意味着一次故障只会恶化延迟，而不是可用性。
- **安全（07）：** 补全跑在用户代码上，所以要把 prompt 当作不可信输入，并且不要记录原始的专有代码（与 08/01 的安全日志相关）。

## 第 5 步 —— 追问（每一个都是杠杆，不是重新设计）

面试官会加压。因为是你定的容量，每个回答都是在已有数字上的一步棋：

| 他说…… | 你伸手去拿……（那个杠杆） |
|---|---|
| 「现在有 10 万用户。」 | 需求 ×10 → GPU 约 ×10（第 2 步是线性放大的）；同时检查队列/网关层是否也要跟着扩。 |
| 「成本砍一半。」 | 量化模型（09/02）→ 更少/更便宜的 GPU；再加上更激进的非峰值降容。点名质量检查（02/04）。 |
| 「p99 TTFT 飙了。」 | 对交互式服务来说 batch size 太高，或者某个副本过热。限制 batch、加余量、查队列深度：就是第 4 步的瓶颈。 |
| 「上下文更长（整个文件）。」 | KV cache 变大 → batch 变小 → 吞吐下降 → 需要更多 GPU（09/02 的练习）。这是容量决策，不是白捡的。 |
| 「多区域 / 更低的全球延迟。」 | 每个区域复制一套，前面加地理路由；网关这层缝隙让这件事是叠加，而不是重写。 |

> **🚩 常见错误 ——** 把每个追问都当作一份新设计。它们只是*同一个*已定容量的系统的扰动。把一切都重画一遍的候选人，看起来像是从没搞懂哪个数字驱动哪个框。你已经把数算过了，所以只需拨动受影响的那个杠杆。

## 一页纸回顾

整套方法，压缩成你真正会留在白板上的东西：

```
1. 需求        交互式？延迟目标？QPS？成本上限？质量门槛？
2. 定容量      QPS → tok/s → GPU 数 = 需求 ÷（tok/s/gpu × 利用率）;  VRAM 检查;  $ / 月
3. 架构        客户端 → 网关 → 队列 → vLLM 副本（自动扩缩）→ 缓存;  + 可观测性/评测
4. 瓶颈        突发下的 TTFT · 模型 = 成本/质量旋钮 · 自动扩缩 = 成本/延迟
5. 追问        每一个都是已有数字上的一个杠杆 —— 不是重新设计
```

> **⭐ 关键结论 ——** 你从来不缺零件；你在 00–09 节里把它们都造出来了。系统设计就是那一轮，让你在压力下把它们*组合*起来、以需求和数据开头、并为取舍辩护。这种组合，讲清楚了，就是 AI 系统岗的录用信号。

## 练习

1. **换题目，留方法。** 换一个*不同的*服务，把五步全部重跑一遍：「为一个客服 RAG 聊天机器人设计后端，500 个并发用户。」记下数字和架构在哪里与编程助手分道扬镳（提示：更长的上下文、多一跳检索、吞吐比 TTFT 更重要）。
2. **为一个数字辩护。** 有人质疑「为什么 2 张 GPU，不是 1 张？」写两句回答，既要引用你第 2 步的算术，*也*要给出冗余的理由。这正是这一轮考的对话。
3. **找出你自己的瓶颈。** 对你计划中的 capstone（第 12 节），只做第 1–2 步，然后指出在 10 倍负载下最先崩的单一资源，以及你会拉的那个杠杆。一段话。
4. **成本砍半。** 拿那个推演过的例子，做出一版月成本大致*减半*的方案。列出你拉动的每一个杠杆，以及每个杠杆你接受的风险，然后说明你会先动哪一个、为什么。